# Module `algorithms/grasp.py`

GRASP signifie **Greedy Randomized Adaptive Search Procedure** (procedure de recherche adaptative gloutonne et randomisee), introduite par Feo & Resende [1]. C'est une **metaheuristique multi-departs** : on construit beaucoup de solutions differentes (chacune par une heuristique partiellement aleatoire), on les ameliore localement, et on garde la meilleure rencontree.

**Pourquoi cette structure plutot qu'une seule descente ?** Parce qu'une heuristique purement gloutonne (toujours choisir le meilleur voisin local) est **deterministe** et tombe systematiquement dans le **meme** optimum local. Or pour le VRP, cet optimum est presque toujours mediocre : la decision gloutonne ignore les consequences a long terme (un client tres proche choisi tot peut isoler des clients eloignes). GRASP perturbe la phase de construction pour explorer **plusieurs bassins d'attraction** differents, et la recherche locale fait descendre chacun d'eux jusqu'a son fond.

Chaque iteration enchaine donc deux phases :

1. **Construction gloutonne randomisee** via une *Restricted Candidate List* (RCL = liste restreinte des candidats) : a chaque etape on tire au hasard parmi les meilleurs candidats, pas seulement le meilleur.
2. **Recherche locale** via `local_search` (relocate + swap + 2-opt) : on descend deterministe jusqu'a un optimum local du voisinage.

Sur `max_iterations` redemarrages independants, on retient la meilleure solution observee. Cette structure simple est detaillee dans le livre de reference de Resende & Ribeiro [2].


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.algorithms import grasp, greedy_randomized_construction, local_search, total_cost
from cesipath.solver_input import build_static_solver_input
import random


## 1. Instance de travail


In [2]:
cfg = GraphGenerationConfig(node_count=15, seed=3)
instance = GraphGenerator(cfg).generate()
si = build_static_solver_input(instance)
print(f"n = {instance.node_count}, capacite = {si.vehicle_capacity}, depot = {si.depot}")


n = 15, capacite = 40, depot = 0


## 2. Parametre cle : `rcl_alpha`

A chaque etape de la construction, on regarde les clients **admissibles** (ceux dont la demande tient dans la capacite residuelle du vehicule) et on les trie par proximite au noeud courant - heuristique du **plus proche voisin**, qui est la regle gloutonne classique pour le TSP/VRP (deja utilisee par Clarke & Wright [3] dans leur algorithme historique des economies). La RCL contient ensuite les `max(1, round(len(feasible) * rcl_alpha))` meilleurs. Le prochain client est tire **uniformement** dans la RCL.

`rcl_alpha` est donc un curseur **glouton <-> aleatoire** :

- `rcl_alpha = 0` -> RCL de taille 1 -> construction **deterministe** (plus proche voisin pur).
- `rcl_alpha = 1` -> RCL = tous les admissibles -> construction **completement aleatoire**.
- Valeurs usuelles : **0.2 a 0.4** pour avoir de la diversite sans trop se degrader. Cette plage est confirmee par les etudes empiriques de Resende & Ribeiro [2, chap. 2] et par les analyses de Festa & Resende [4, section 4].

**Pourquoi pas un alpha proche de 0 ?** Parce qu'on perdrait l'interet du multi-depart : toutes les iterations partiraient quasiment du meme bassin. **Pourquoi pas un alpha proche de 1 ?** Parce que les solutions construites seraient si mauvaises (couts moyens 50% au-dessus du glouton, voir cellule suivante) que la recherche locale a un cout fixe ne suffirait pas a rattraper le retard.

L'experience ci-dessous mesure ces effets : on construit 20 solutions (sans recherche locale) pour chaque alpha et on observe la dispersion.


In [3]:
rng = random.Random(0)

for alpha in (0.0, 0.2, 0.5, 1.0):
    costs = []
    r = random.Random(0)
    for _ in range(20):
        routes = greedy_randomized_construction(si, alpha, r)
        costs.append(round(total_cost(routes, si.cost_matrix), 2))
    print(f"alpha={alpha:>3} : cout min={min(costs):>6.2f}  max={max(costs):>6.2f}  moy={sum(costs)/len(costs):.2f}")


alpha=0.0 : cout min=772.37  max=772.37  moy=772.37
alpha=0.2 : cout min=801.63  max=960.20  moy=865.50
alpha=0.5 : cout min=857.87  max=1101.68  moy=976.74
alpha=1.0 : cout min=957.08  max=1317.56  moy=1149.63


## 3. Construction puis polissage

La construction seule (meme la deterministe a `alpha=0`) donne une solution **structurellement correcte** mais qui contient quasi toujours des inefficacites locales : un client mal place dans une tournee, deux aretes qui se croisent, etc. C'est inevitable car le glouton prend ses decisions **sans pouvoir revenir en arriere**.

La phase de **recherche locale** (`local_search`, voir le notebook `neighborhood.ipynb`) compose trois operateurs deterministes :
- **relocate** : deplacer un client d'une tournee a une autre,
- **swap** : echanger deux clients entre tournees,
- **2-opt** [5] : inverser un segment dans une tournee, ce qui denoue les croisements.

Elle s'arrete quand plus aucun mouvement n'ameliore : on a alors un **optimum local** au sens du voisinage compose. Le gain typique sur une construction GRASP est de l'ordre de **10 a 30%** (voir Resende & Ribeiro [2, chap. 4] sur le role de la recherche locale dans GRASP). La cellule ci-dessous le mesure sur cette instance.


In [4]:
rng = random.Random(7)
routes = greedy_randomized_construction(si, rcl_alpha=0.3, rng=rng)
before = total_cost(routes, si.cost_matrix)
polished = local_search(routes, si.cost_matrix, si.demands, si.vehicle_capacity)
after = total_cost(polished, si.cost_matrix)
print(f"construction  : {before:.2f}  ({len(routes)} tournees)")
print(f"local_search  : {after:.2f}  ({len(polished)} tournees)   gain={100*(before-after)/before:.1f} %")


construction  : 845.45  (3 tournees)
local_search  : 699.07  (3 tournees)   gain=17.3 %


## 4. GRASP complet

`grasp(solver_input, max_iterations, rcl_alpha, use_local_search, seed)` retourne une `VRPSolution` contenant la meilleure solution rencontree.

**Pourquoi `max_iterations=100` par defaut ?** GRASP n'a pas de critere d'arret intrinseque (contrairement a SA qui peut s'arreter quand la temperature gele) : on doit fixer un budget. Empiriquement, sur les VRP de taille moyenne, la meilleure solution converge en quelques dizaines d'iterations - ce qu'on observe dans la cellule ci-dessous, ou passer de 20 a 50 iterations n'apporte plus rien sur cette instance. Festa & Resende [4, section 5] notent qu'un budget de 50 a quelques centaines d'iterations est typique pour GRASP.

Le flag `use_local_search=False` permet d'observer GRASP sans intensification (peu recommande en pratique, voir section suivante).


In [5]:
for it in (5, 20, 50):
    sol = grasp(si, max_iterations=it, rcl_alpha=0.3, seed=42)
    print(f"max_iterations={it:>2} : cout={sol.total_cost:.2f}  routes={sol.route_count}")


max_iterations= 5 : cout=736.19  routes=3
max_iterations=20 : cout=699.07  routes=3
max_iterations=50 : cout=699.07  routes=3


## 5. Impact du `use_local_search`

**Pourquoi est-ce essentiel ?** Sans `local_search`, GRASP devient un simple **echantillonnage aleatoire biaise** de l'espace des solutions : on tire 30 solutions construites a coup de RCL, on garde la meilleure - mais aucune n'a ete optimisee. La phase de **selection des meilleures** ne peut jouer son role que si on lui presente des solutions deja amenees au fond de leur bassin d'attraction.

C'est exactement la dichotomie **diversification / intensification** centrale dans toutes les metaheuristiques (terminologie de Glover & Kochenberger [6, chap. 1]) :
- **diversification** : explorer des regions differentes de l'espace -> ici la randomisation RCL,
- **intensification** : exploiter chaque region en y descendant a fond -> ici la `local_search`.

Sans intensification, GRASP n'est plus que la moitie de lui-meme. Le test ci-dessous mesure l'ecart : sans `local_search` on perd typiquement 15 a 20% sur cette instance.


In [6]:
sol_ls   = grasp(si, max_iterations=30, rcl_alpha=0.3, use_local_search=True,  seed=1)
sol_nols = grasp(si, max_iterations=30, rcl_alpha=0.3, use_local_search=False, seed=1)
print(f"avec    local_search : {sol_ls.total_cost:.2f}")
print(f"sans    local_search : {sol_nols.total_cost:.2f}")
print(f"amelioration relative: {100*(sol_nols.total_cost - sol_ls.total_cost)/sol_nols.total_cost:.1f} %")


avec    local_search : 699.07
sans    local_search : 826.61
amelioration relative: 15.4 %


## 6. Reproductibilite

Tous les tirages aleatoires (RCL et eventuels mouvements stochastiques) passent par un unique `random.Random(seed)` cree au debut de l'algorithme. Aucun appel ne touche le `random` global. Consequence : a `seed` egal, l'algorithme produit **exactement** la meme trajectoire et donc la meme solution finale - propriete indispensable pour le debug et pour comparer equitablement plusieurs algorithmes dans `benchmark.py`.


In [7]:
a = grasp(si, max_iterations=30, rcl_alpha=0.3, seed=123)
b = grasp(si, max_iterations=30, rcl_alpha=0.3, seed=123)
print("identiques ?", a.total_cost == b.total_cost and a.routes == b.routes)


identiques ? True


## 7. Quand choisir GRASP ?

**Forces** :

- **Budget faible a moyen** : quelques dizaines d'iterations donnent deja une bonne solution.
- **Code simple a lire et a maintenir** : pas d'etat temporel a regler (comme la temperature de SA) ni de memoire (comme la liste tabou). Deux parametres seulement (`rcl_alpha`, `max_iterations`).
- **Bon baseline** : GRASP est traditionnellement le point de depart auquel comparer les autres metaheuristiques (Festa & Resende [4, intro]).
- **Parallelisation triviale** : chaque iteration est independante, donc embarrassingly parallel.

**Limites** :

- **Pas de memoire entre iterations** : chaque redemarrage ignore l'historique, donc GRASP peut redescendre plusieurs fois dans le meme bassin d'attraction. Le tabou (memoire courte) ou le genetique (population) sont plus diversifiants.
- **Pas d'apprentissage de l'alpha** : la version reactive `Reactive GRASP` [7] ajuste `alpha` en cours de route, mais elle n'est pas implementee ici par simplicite.

---

## References

[1] **Feo, T. A. & Resende, M. G. C. (1995).** *Greedy randomized adaptive search procedures.* Journal of Global Optimization, 6(2), 109-133. https://doi.org/10.1007/BF01096763

[2] **Resende, M. G. C. & Ribeiro, C. C. (2016).** *Optimization by GRASP: Greedy Randomized Adaptive Search Procedures.* Springer. https://doi.org/10.1007/978-1-4939-6530-4

[3] **Clarke, G. & Wright, J. W. (1964).** *Scheduling of vehicles from a central depot to a number of delivery points.* Operations Research, 12(4), 568-581. https://doi.org/10.1287/opre.12.4.568

[4] **Festa, P. & Resende, M. G. C. (2009).** *An annotated bibliography of GRASP - Part I: Algorithms.* International Transactions in Operational Research, 16(1), 1-24. https://doi.org/10.1111/j.1475-3995.2009.00663.x

[5] **Lin, S. (1965).** *Computer solutions of the traveling salesman problem.* Bell System Technical Journal, 44(10), 2245-2269. https://doi.org/10.1002/j.1538-7305.1965.tb04146.x

[6] **Glover, F. & Kochenberger, G. A. (eds.) (2003).** *Handbook of Metaheuristics.* Springer (International Series in Operations Research & Management Science, vol. 57). https://doi.org/10.1007/b101874

[7] **Prais, M. & Ribeiro, C. C. (2000).** *Reactive GRASP: An application to a matrix decomposition problem in TDMA traffic assignment.* INFORMS Journal on Computing, 12(3), 164-176. https://doi.org/10.1287/ijoc.12.3.164.12639
